# Lecture 5: Calculators and Computing Properties

## Overview
**Questions**
- How do I attach a calculator to an `Atoms` object?
- What properties can a calculator compute?
- How do I use file-based calculators (e.g. Quantum ESPRESSO)?
- How do I cache and store results?

**Objectives**
- Attach an EMT calculator and compute energy, forces, and stress
- Understand the calculator interface
- Learn about file-based calculators and their workflow
- Store results using ASE databases


## Attaching a Calculator

A calculator is attached to an `Atoms` object with `atoms.calc = Calculator()`. After that, any call to `get_potential_energy()`, `get_forces()`, or `get_stress()` will trigger the calculation.


In [ ]:
from ase.build import bulk
from ase.calculators.emt import EMT
import numpy as np
import matplotlib.pyplot as plt

# Build FCC aluminium
al = bulk('Al', 'fcc', a=4.05)
al.calc = EMT()

# Compute properties
energy = al.get_potential_energy()
forces = al.get_forces()
stress = al.get_stress()

print(f"Energy:  {energy:.4f} eV")
print(f"Forces (should be ~0 at equilibrium):\n{forces}")
print(f"Stress tensor (Voigt, eV/Å³): {stress}")


## Equation of State

A fundamental calculation is the **equation of state (EOS)** — the relationship between volume and energy. It lets us extract the equilibrium lattice constant, bulk modulus, and cohesive energy.

This is directly useful for quantum optics materials: strain in a crystal shifts emission wavelengths, so knowing the bulk modulus is important for device design.


In [ ]:
from ase.eos import EquationOfState

# Compute E-V curve for Al
volumes, energies = [], []
a0 = 4.05
for scale in np.linspace(0.94, 1.06, 15):
    al_scaled = bulk('Al', 'fcc', a=a0 * scale)
    al_scaled.calc = EMT()
    volumes.append(al_scaled.get_volume())
    energies.append(al_scaled.get_potential_energy())

# Fit a Birch-Murnaghan EOS
eos = EquationOfState(volumes, energies, eos='birchmurnaghan')
v0, e0, B = eos.fit()

print(f"Equilibrium volume V₀ = {v0:.3f} Å³")
print(f"Equilibrium energy E₀ = {e0:.4f} eV")
print(f"Bulk modulus B = {B / 1.6022e-19 * 1e30 * 1e-9:.1f} GPa")

fig = eos.plot(show=False)
plt.title('Equation of State — FCC Al (EMT)')
plt.tight_layout()
plt.show()


## File-based Calculators

For real research, we use **density functional theory (DFT)** via file-based calculators. The workflow is:

1. ASE writes an input file (e.g. `pw.in` for Quantum ESPRESSO)
2. ASE calls the external code as a subprocess
3. ASE reads the output file and parses results

### Quantum ESPRESSO example

```python
from ase.calculators.espresso import Espresso

pseudopotentials = {'Ga': 'Ga.pbe-dn-kjpaw_psl.1.0.0.UPF',
                    'N': 'N.pbe-n-kjpaw_psl.1.0.0.UPF'}

calc = Espresso(
    pseudopotentials=pseudopotentials,
    pseudo_dir='/path/to/pseudopotentials/',
    input_data={
        'system': {
            'ecutwfc': 60,       # plane-wave cutoff (Ry)
            'ecutrho': 480,      # charge density cutoff
            'occupations': 'smearing',
            'smearing': 'cold',
            'degauss': 0.01,
        },
        'electrons': {
            'conv_thr': 1e-8,
        },
    },
    kpts=(6, 6, 4),              # k-point mesh
)

gan.calc = calc
energy = gan.get_potential_energy()   # triggers DFT calculation
```

> **Note:** Running this requires Quantum ESPRESSO to be installed and pseudopotential files. In this course we use the EMT potential for interactive exercises.


## K-point Convergence

For periodic systems, we sample the Brillouin zone with a **k-point mesh**. Finer meshes give more accurate results but cost more compute time. Always test convergence!


In [ ]:
from ase.build import bulk
from ase.calculators.emt import EMT   # proxy for DFT in this demo

al = bulk('Al', 'fcc', a=4.05)

# Simulate k-point convergence by sampling different mesh sizes.
# In a real DFT calculation you would use:
# calc = Espresso(..., kpts=(k, k, k))
# Here we just demonstrate the workflow.

k_values = [2, 4, 6, 8, 10, 12]
energies_k = []

for k in k_values:
    # For EMT energy doesn't depend on k-points, but this shows the pattern
    al.calc = EMT()
    energies_k.append(al.get_potential_energy())

# In a real DFT run this plot would show convergence
plt.figure(figsize=(7, 4))
plt.plot(k_values, energies_k, 'o-', color='crimson')
plt.xlabel('k-point mesh (N×N×N)')
plt.ylabel('Energy (eV)')
plt.title('K-point convergence test (EMT proxy)')
plt.tight_layout()
plt.show()

print("In real DFT, look for convergence to ~1 meV/atom.")
print("For quantum defect calculations, very fine k-meshes are needed")
print("near the Γ-point where defect states often appear.")


## Storing Results with ASE Database

The **ASE database** (`ase.db`) is a lightweight SQLite-based database for storing atoms and their calculated properties. It's excellent for managing screening calculations.


In [ ]:
import ase.db

# Create a database
db = ase.db.connect('/tmp/materials.db')

# Store some structures with calculated properties
structures = {
    'Al': bulk('Al', 'fcc', a=4.05),
    'C_diamond': bulk('C', 'diamond', a=3.57),
    'GaN': bulk('GaN', 'wurtzite', a=3.19, c=5.19),
}

for name, atoms in structures.items():
    atoms.calc = EMT()
    energy = atoms.get_potential_energy()
    db.write(atoms, name=name,
             energy_per_atom=energy/len(atoms),
             material_class='quantum_optics_host')

# Query the database
print("Database contents:")
for row in db.select():
    print(f"  {row.name:12s}: E/atom = {row.energy_per_atom:.4f} eV, "
          f"formula = {row.formula}")


## Key Points

- Attach a calculator with `atoms.calc = Calculator()`
- `get_potential_energy()`, `get_forces()`, `get_stress()` trigger calculations
- The EOS gives equilibrium lattice constant and bulk modulus
- File-based calculators write/read input/output files for codes like QE or VASP
- Always test k-point and cutoff convergence in DFT
- Use `ase.db` to store and query results

## Exercise 5.1

Compute the equation of state for diamond carbon using EMT and extract the bulk modulus. Compare to the experimental value (~440 GPa). Note: EMT is not accurate for carbon, so this is just for practice!

## Exercise 5.2

Read about the `GPAW` calculator in the ASE documentation. GPAW is a DFT code written in Python that integrates tightly with ASE. What types of calculations is it particularly well-suited for?
